# 特征筛选模块演示 (Selectors Module)

本notebook演示hscredit库中特征筛选模块的全部功能，包含多种筛选方法。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n目标列分布:\n{df['FPD15'].value_counts()}")

/Users/xiaoxi/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据形状: (12448, 85)

列名: ['MOB1', 'MOB2', 'loan_date', 'bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_confidence', 'consumer_finance_loan_credit_line', 'consumer_finance_loan_credit_line_avg', 'consumer_finance_loan_credit_line_max', 'consumer_finance_loan_lender_count', 'consumer_finance_loan_product_count', 'credit_loan_duration', 'hit_consumer_finance_lender_count', 'hit_lender_count', 'hit_network_loan_lender_count', 'last_performance_days', 'lender_count_12m', 'lender_count_1m', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_1m', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_coun

In [2]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")

# 准备数据
X = df[feature_cols]
y = df[target_col]
print(f"X形状: {X.shape}, y形状: {y.shape}")

特征数量: 80
X形状: (12448, 80), y形状: (12448,)


## 1. 导入特征筛选模块

筛选器支持独立使用和Pipeline集成。

In [3]:
from hscredit.core.selectors import (
    VarianceSelector,
    NullSelector,
    CorrSelector,
    VIFSelector,
    IVSelector,
    PSISelector,
    LiftSelector,
    CardinalitySelector,
    TypeSelector,
    FeatureImportanceSelector,
    CompositeFeatureSelector
)
print("所有筛选器已导入成功！")

所有筛选器已导入成功！


## 2. 方差筛选器 (VarianceSelector)

剔除方差低于阈值的特征。

In [4]:
# 方差筛选演示
selector_var = VarianceSelector(threshold=0.01)
X_var = selector_var.fit_transform(X, y)
print(f"方差筛选前特征数: {X.shape[1]}")
print(f"方差筛选后特征数: {X_var.shape[1]}")
print(f"剔除的特征: {selector_var.removed_features_}")

方差筛选前特征数: 80
方差筛选后特征数: 76
剔除的特征: ['normal_repayment_order_in_total_order_rate', 'overdue_loan_m1_count_12m', 'overdue_loan_m1_count_24m', 'overdue_loan_m1_count_6m']


## 3. 缺失率筛选器 (NullSelector)

剔除缺失率高于阈值的特征。

In [5]:
# 缺失率筛选演示
selector_null = NullSelector(threshold=0.5)
X_null = selector_null.fit_transform(X, y)
print(f"缺失率筛选前特征数: {X.shape[1]}")
print(f"缺失率筛选后特征数: {X_null.shape[1]}")
print(f"\n各特征缺失率:")
null_rates = X.isnull().mean()
print(null_rates[null_rates > 0].sort_values(ascending=False))

缺失率筛选前特征数: 80
缺失率筛选后特征数: 78

各特征缺失率:
dxm_v6pro   0.8181
bcf_fpv1    0.8181
dtype: float64


## 4. 相关性筛选器 (CorrSelector)

剔除高度相关的特征，保留与目标相关性更高的特征。

In [6]:
# 相关性筛选演示
selector_corr = CorrSelector(threshold=0.8)
X_corr = selector_corr.fit_transform(X, y)
print(f"相关性筛选前特征数: {X.shape[1]}")
print(f"相关性筛选后特征数: {X_corr.shape[1]}")
print(f"\n剔除的特征: {selector_corr.removed_features_}")

相关性筛选前特征数: 80
相关性筛选后特征数: 36

剔除的特征: ['performance_loan_sum_3m', 'hit_lender_count', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_count_12m', 'loan_amt_between_3k_10k_count_12m', 'consumer_finance_loan_product_count', 'consumer_finance_loan_lender_count', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_credit_line', 'overdue_loan_m0_count_6m', 'performance_loan_count_12m', 'performance_loan_count_1m', 'performance_loan_count_24m', 'performance_loan_count_3m', 'performance_loan_count_6m', 'performance_loan_sum_12m', 'performance_loan_sum_1m', 'performance_loan_sum_24m', 'overdue_amt_sum_12m', 'loan_amt_sum_24m', 'loan_count_3m', 'loan_behavior_confidence', 'loan_behavior_score', 'loan_count_12m', 'loan_count_1m', 'loan_count_24m', 'loan_count_6m', '

## 5. VIF筛选器 (VIFSelector)

剔除方差膨胀因子(VIF)过高的特征，用于检测多重共线性。

In [7]:
# VIF筛选演示（使用数值特征）
numeric_X = X.select_dtypes(include=[np.number])
selector_vif = VIFSelector(threshold=10.0, n_jobs=-1)
X_vif = selector_vif.fit_transform(numeric_X, y)
print(f"VIF筛选前特征数: {numeric_X.shape[1]}")
print(f"VIF筛选后特征数: {X_vif.shape[1]}")
print(f"\n各特征VIF值:")
vif_scores = selector_vif.scores_
print(vif_scores.sort_values(ascending=False))

VIF筛选前特征数: 80
VIF筛选后特征数: 52

各特征VIF值:
performance_loan_count_1m                    9.2837
lender_count_6m                              9.0797
loan_count_3m                                9.0720
lender_inquiry_count                         8.9303
lender_inquiry_count_3m                      8.2837
bcf_fpv1                                     8.2292
performance_loan_sum_6m                      8.0240
dxm_v6pro                                    7.8031
overdue_amt_sum_12m                          7.7553
consumer_finance_loan_credit_line_avg        7.2730
performance_loan_sum_1m                      7.1175
performance_loan_sum_24m                     6.7617
overdue_amt_sum_6m                           6.6609
consumer_finance_loan_confidence             6.5930
overdue_amt_sum_24m                          6.5908
performance_loan_count_24m                   6.4399
overdue_loan_m0_count_12m                    6.1069
network_loan_lender_count_24m                6.0629
loan_behavior_confidence  

## 6. IV筛选器 (IVSelector)

基于信息值(IV)筛选特征，IV越高特征预测能力越强。

In [8]:
# IV筛选演示
selector_iv = IVSelector(threshold=0.02)
X_iv = selector_iv.fit_transform(X, y)
print(f"IV筛选前特征数: {X.shape[1]}")
print(f"IV筛选后特征数: {X_iv.shape[1]}")
print(f"\n各特征IV值（从高到低）:")
iv_scores = selector_iv.scores_
print(iv_scores.sort_values(ascending=False))

IV筛选前特征数: 80
IV筛选后特征数: 69

各特征IV值（从高到低）:
dxm_v6pro                               25.5116
bcf_fpv1                                13.8772
consumer_finance_loan_credit_line_avg    4.9024
bairong_v1                               4.3981
credit_loan_duration                     1.0703
                                          ...  
overdue_loan_m0_count_24m                0.0104
overdue_loan_m0_count_6m                 0.0044
overdue_loan_m1_count_12m                0.0034
overdue_loan_m1_count_24m                0.0011
overdue_loan_m1_count_6m                 0.0000
Length: 80, dtype: float64


## 7. PSI筛选器 (PSISelector)

基于群体稳定性指数(PSI)筛选特征。

In [9]:
# PSI筛选演示
# 将数据分为训练集和测试集来计算PSI
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(X, test_size=0.3, random_state=42)
y_train, y_test = train_test_split(y, test_size=0.3, random_state=42)

selector_psi = PSISelector(threshold=0.1)
selector_psi.fit(X_train, y_train)
X_psi = selector_psi.transform(X_test)
print(f"PSI筛选后特征数: {X_psi.shape[1]}")
print(f"\n各特征PSI值:")
psi_scores = selector_psi.scores_
print(psi_scores.sort_values(ascending=False))

PSI筛选后特征数: 80

各特征PSI值:
lender_inquiry_count_6m            0.0100
performance_loan_count_1m          0.0098
loan_amt_between_1k_3k_count_12m   0.0090
lender_inquiry_count_1m            0.0086
bj_qy24                            0.0085
                                    ...  
overdue_loan_m1_count_12m          0.0000
overdue_loan_m0_count_6m           0.0000
overdue_loan_m0_count_12m          0.0000
overdue_amt_sum_12m                0.0000
overdue_amt_sum_6m                 0.0000
Length: 80, dtype: float64


## 8. Lift筛选器 (LiftSelector)

基于Lift值筛选特征，评估特征对坏样本的区分能力。

In [10]:
# Lift筛选演示
selector_lift = LiftSelector(threshold=2)
X_lift = selector_lift.fit_transform(X, y)
print(f"Lift筛选前特征数: {X.shape[1]}")
print(f"Lift筛选后特征数: {X_lift.shape[1]}")
print(f"\n各特征Lift值:")
lift_scores = selector_lift.scores_
print(lift_scores.sort_values(ascending=False))

Lift筛选前特征数: 80
Lift筛选后特征数: 61

各特征Lift值:
bj_qy24                                 15.0702
consumer_finance_loan_credit_line_max   15.0702
lender_count_3m                         15.0702
lender_inquiry_count                    15.0702
lender_inquiry_count_1m                 15.0702
                                          ...  
loan_amt_sum_1m                          1.3725
network_loan_confidence                  1.0715
network_loan_lender_count_12m            1.0568
overdue_loan_m1_count_24m                1.0012
overdue_loan_m1_count_6m                 1.0000
Length: 80, dtype: float64


## 9. 基数筛选器 (CardinalitySelector)

剔除基数过高或过低的特征。

In [11]:
# 基数筛选演示
selector_card = CardinalitySelector(threshold=20)
X_card = selector_card.fit_transform(X, y)
print(f"基数筛选前特征数: {X.shape[1]}")
print(f"基数筛选后特征数: {X_card.shape[1]}")

iv_scores = selector_iv.scores_
print(iv_scores.sort_values(ascending=False))

print(f"\n剔除的特征: {selector_corr.removed_features_}")

基数筛选前特征数: 80
基数筛选后特征数: 28
dxm_v6pro                               25.5116
bcf_fpv1                                13.8772
consumer_finance_loan_credit_line_avg    4.9024
bairong_v1                               4.3981
credit_loan_duration                     1.0703
                                          ...  
overdue_loan_m0_count_24m                0.0104
overdue_loan_m0_count_6m                 0.0044
overdue_loan_m1_count_12m                0.0034
overdue_loan_m1_count_24m                0.0011
overdue_loan_m1_count_6m                 0.0000
Length: 80, dtype: float64

剔除的特征: ['performance_loan_sum_3m', 'hit_lender_count', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_count_12m', 'loan_amt_between_3k_10k_count_12m', 'consumer_finance_loan_product_count', 'consumer_finance_loan_lender_count', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m'

## 10. 类型筛选器 (TypeSelector)

按数据类型筛选特征。

In [12]:
# 类型筛选演示
selector_type = TypeSelector(dtype_include=['number'])
X_type = selector_type.fit_transform(X, y)
print(f"数值类型特征数: {X_type.shape[1]}")

selector_type_obj = TypeSelector(dtype_include=['object'])
X_type_obj = selector_type_obj.fit_transform(X, y)
print(f"对象类型特征数: {X_type_obj.shape[1]}")

数值类型特征数: 80
对象类型特征数: 0


## 11. 特征重要性筛选器 (FeatureImportanceSelector)

基于模型特征重要性筛选特征。

In [13]:
# 特征重要性筛选演示
from sklearn.ensemble import RandomForestClassifier

selector_imp = FeatureImportanceSelector(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    threshold=0.01
)
X_imp = selector_imp.fit_transform(X, y)
print(f"重要性筛选前特征数: {X.shape[1]}")
print(f"重要性筛选后特征数: {X_imp.shape[1]}")
print(f"\n各特征重要性（从高到低）:")
importance_scores = selector_imp.scores_
print(importance_scores.sort_values(ascending=False))

重要性筛选前特征数: 80
重要性筛选后特征数: 49

各特征重要性（从高到低）:
bairong_v1                      0.0366
xiaoniu_c3                      0.0287
bj_qy24                         0.0277
lender_inquiry_count_1m         0.0251
lender_inquiry_count_6m         0.0247
                                 ...  
network_loan_confidence         0.0011
network_loan_lender_count_12m   0.0009
overdue_loan_m1_count_12m       0.0004
overdue_loan_m1_count_24m       0.0001
overdue_loan_m1_count_6m        0.0000
Length: 80, dtype: float64


## 12. 组合筛选器 (CompositeFeatureSelector)

将多个筛选器组合使用，按顺序执行筛选。

In [16]:
# 组合筛选演示
composite_selector = CompositeFeatureSelector(
    selectors=[
        ('null', NullSelector(threshold=0.8)),
        ('variance', VarianceSelector(threshold=0.01)),
        ('iv', IVSelector(threshold=0.02)),
        ('correlation', CorrSelector(threshold=0.8))
    ],
    strategy='sequential'  # 或 'union', 'intersection'
)

X_composite = composite_selector.fit_transform(X, y)
print(f"组合筛选前特征数: {X.shape[1]}")
print(f"组合筛选后特征数: {X_composite.shape[1]}")

# 查看筛选报告
report = composite_selector.get_selection_report_df()
# pd.DataFrame(report['剔除详情'])
composite_selector.get_selection_report_df()

组合筛选前特征数: 80
组合筛选后特征数: 21


,特征,筛选轮次,筛选器,筛选器类型,策略,状态,得分,剔除原因,该轮输入特征数,该轮输出特征数,该轮剔除特征数,阈值
0,apply_for_admission_confidence,1,null,NullSelector,sequential,选中,0.0000,,80,78,2,NaN
1,apply_for_admission_score,1,null,NullSelector,sequential,选中,0.0000,,80,78,2,NaN
2,bairong_v1,1,null,NullSelector,sequential,选中,0.0000,,80,78,2,NaN
3,bcf_fpv1,1,null,NullSelector,sequential,剔除,0.8181,缺失率(81.81%) >= 阈值(80.00%),80,78,2,NaN
4,bj_qy24,1,null,NullSelector,sequential,选中,0.0000,,80,78,2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
297,xiaoniu_c3,4,correlation,CorrSelector,sequential,选中,1.0000,,66,21,45,NaN
298,[SUMMARY],1,null,NullSelector,sequential,汇总,,,80,78,2,0.8000
299,[SUMMARY],2,variance,VarianceSelector,sequential,汇总,,,78,74,4,0.0100
300,[SUMMARY],3,iv,IVSelector,sequential,汇总,,,74,66,8,0.0200


In [18]:
# 查看详细筛选报告（穿透底层筛选器）
detailed_report = composite_selector.get_selection_report_df()
print(f"详细报告总行数: {len(detailed_report)}")

# 显示特征追踪信息（排除汇总行）
feature_trace = detailed_report[detailed_report['特征'] != '[SUMMARY]']
print(f"特征追踪行数: {len(feature_trace)}")
print("特征追踪示例（前10行）：")
feature_trace.head(10)

# 显示汇总信息
print("各轮筛选汇总：")
summary = detailed_report[detailed_report['特征'] == '[SUMMARY]']
summary[['筛选轮次', '筛选器', '该轮输入特征数', '该轮输出特征数', '该轮剔除特征数']]

详细报告总行数: 302
特征追踪行数: 298
特征追踪示例（前10行）：
各轮筛选汇总：


,筛选轮次,筛选器,该轮输入特征数,该轮输出特征数,该轮剔除特征数
298,1,null,80,78,2
299,2,variance,78,74,4
300,3,iv,74,66,8
301,4,correlation,66,21,45


## 13. 筛选结果汇总

对比不同筛选方法的结果。

In [19]:
# 筛选方法对比
results = {
    '原始特征': X.shape[1],
    '方差筛选': X_var.shape[1],
    '缺失率筛选': X_null.shape[1],
    '相关性筛选': X_corr.shape[1],
    'IV筛选': X_iv.shape[1],
    'Lift筛选': X_lift.shape[1],
    '重要性筛选': X_imp.shape[1],
    '组合筛选': X_composite.shape[1]
}

results_df = pd.DataFrame(list(results.items()), columns=['筛选方法', '保留特征数'])
results_df['淘汰特征数'] = X.shape[1] - results_df['保留特征数']
results_df['保留比例'] = (results_df['保留特征数'] / X.shape[1] * 100).round(2).astype(str) + '%'
print(results_df.to_string(index=False))

  筛选方法  保留特征数  淘汰特征数   保留比例
  原始特征     80      0 100.0%
  方差筛选     76      4  95.0%
 缺失率筛选     78      2  97.5%
 相关性筛选     36     44  45.0%
  IV筛选     69     11 86.25%
Lift筛选     61     19 76.25%
 重要性筛选     49     31 61.25%
  组合筛选     21     59 26.25%


## 14. 保存筛选结果

将筛选后的特征和报告保存到文件。

In [20]:
# 保存筛选结果
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/selector_results.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='筛选对比', index=False)
    iv_scores.sort_values(ascending=False).to_frame('IV值').to_excel(writer, sheet_name='IV分数')
    importance_scores.sort_values(ascending=False).to_frame('重要性').to_excel(writer, sheet_name='重要性分数')

print(f"筛选结果已保存至: {output_path}")

筛选结果已保存至: /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/selector_results.xlsx
